# Experimentos - MLP do Zero

Este notebook registra os experimentos do projeto. A implementacao principal esta no pacote `mlp/` e os treinos podem ser executados pelos scripts em `scripts/`.

## Configuracoes comparadas

| Experimento | Arquitetura | Ativacao | Learning rate | Momentum |
| --- | --- | --- | --- | --- |
| baseline_relu | 784-256-128-10 | ReLU | 0.05 | 0.9 |
| deeper_relu | 784-256-128-64-10 | ReLU | 0.03 | 0.9 |

In [ ]:
# Rode esta celula depois de instalar as dependencias.
# Ela executa os dois experimentos definidos em scripts/run_experiments.py.

import subprocess
import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

subprocess.run([sys.executable, '-m', 'scripts.run_experiments'], cwd=project_root, check=True)


In [ ]:
import csv
import json
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

def read_csv(path):
    with path.open(encoding='utf-8') as file:
        return list(csv.DictReader(file))

def print_table(rows, columns):
    if not rows:
        print('Nenhum resultado encontrado. Rode a celula de experimentos primeiro.')
        return
    widths = {col: max(len(col), *(len(str(row.get(col, ''))) for row in rows)) for col in columns}
    print(' | '.join(col.ljust(widths[col]) for col in columns))
    print('-+-'.join('-' * widths[col] for col in columns))
    for row in rows:
        print(' | '.join(str(row.get(col, '')).ljust(widths[col]) for col in columns))

summary_path = project_root / 'results' / 'experiments.csv'
summary_rows = read_csv(summary_path) if summary_path.exists() else []
summary_columns = [
    'run_name', 'layers', 'test_accuracy', 'test_loss',
    'precision_macro', 'recall_macro', 'f1_macro',
    'precision_weighted', 'recall_weighted', 'f1_weighted',
    'balanced_accuracy',
]
print_table(summary_rows, summary_columns)


## Relatorios por digito

As tabelas abaixo mostram precision, recall, F1 e support para cada digito. Recall e especialmente util aqui porque mostra, para cada classe real, quantos exemplos daquele digito a rede conseguiu recuperar corretamente.

In [ ]:
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

for run_name in ['baseline_relu', 'deeper_relu']:
    report_path = project_root / 'results' / f'{run_name}_classification_report.csv'
    print(f'\n{run_name}')
    if report_path.exists():
        rows = read_csv(report_path)
        print_table(rows, ['class', 'precision', 'recall', 'f1', 'support'])
    else:
        print('Relatorio ainda nao encontrado. Rode os experimentos primeiro.')


## Curvas

Depois do treino, abra as imagens em `results/` para analisar se a loss convergiu e se a acuracia de validacao acompanhou a de treino sem overfitting forte.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

for image_name in ['baseline_relu_curves.png', 'baseline_relu_confusion.png']:
    image_path = project_root / 'results' / image_name
    if image_path.exists():
        display(Image(filename=str(image_path)))


## Analise

Preencha apos rodar:

- Qual configuracao atingiu a melhor acuracia?
- A loss caiu de forma consistente?
- Quais digitos aparecem mais confundidos na matriz de confusao?
- A meta de 92% foi atingida?